In [4]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.join(os.getcwd(), '..'))
from constants import DATA_DIR, MODEL_DIR, LOG_DIR

import pycaret
pycaret.__version__

'3.3.2'

In [5]:
continuous = [
    "DataUsed_UL",
    "DataUsed_DL",
    "Speed_UL",
    "Speed_DL",
    "PhyRate",
    "SignalStrength",
    "WifiAssociations",
    "SON_SteerSuccessStatus",
    "KeyProcessCrashes",
    "Latency",
    "HomeWifiScore",
    "PercentageOfDataOn5G",
    "ClientBandwidthUsed",
    "AirtimeCongestion",
    "ActivePhyRate",
    "WifiScores_top3Devices",
]

categorical = [
    "LegacyDevicesExist",
    "StationaryVsMobile",
    "SON_status",
    "ChannelOfOperation",
    "SuddenDropOfConnection",
    "RebootPerformedViaCustomer",
    "StationName",
    "ExtendedBackhaulRange",
    "EthernetAndNot1Gig",
]

discrete = [
    "NumClientsConnected",
    "NumAccessPoints",
    "MeanClientsPerAP",
    "ChannelSwitches",
    "NumDevicePowerCycles",
    "HighSteeringCount",
    "HomesWithWifiCongestion",
    "SON_PerClientSteer",
    "NumIPChanges",
    "NumNeighborsOnSameChannel",
    "NumOverallNeighbors",
    "NumNeighborsOnAdjChannel",
]

In [7]:
# Load synthetic data
data = pd.read_csv(DATA_DIR / 'synthetic_kpi.csv', nrows=5000)
data.head()

,DataUsed_UL,DataUsed_DL,Speed_UL,Speed_DL,LegacyDevicesExist,StationaryVsMobile,WifiScores_top3Devices,PhyRate,SignalStrength,NumClientsConnected,...,ActivePhyRate,SuddenDropOfConnection,RebootPerformedViaCustomer,StationName,NumNeighborsOnSameChannel,NumOverallNeighbors,NumNeighborsOnAdjChannel,NumIPChanges,ExtendedBackhaulRange,EthernetAndNot1Gig
0,224.117296,244.346184,24.886201,170.372869,Yes,Stationary,19.428799,407.362001,-61.579952,7,...,181.195526,No,No,2,3,7,0,1,Yes,Yes
1,121.534834,190.142714,32.904833,151.496648,Yes,Stationary,22.576670,505.233358,-68.115528,8,...,447.587476,No,No,3,6,10,3,0,Yes,No
2,106.712863,303.252176,10.293199,59.872658,No,Mobile,22.830999,598.159580,-55.188275,10,...,198.032127,No,No,9,6,14,3,3,Yes,No
3,187.828692,342.273566,69.956962,119.270685,No,Stationary,23.863197,538.285427,-68.701393,4,...,24.606389,No,No,3,9,12,5,2,No,Yes
4,229.867242,326.287468,27.209818,51.917137,Yes,Stationary,26.248302,688.846932,-56.527905,5,...,217.747655,Yes,Yes,5,10,13,3,1,No,No


# Setup

In [8]:
from pycaret.anomaly import AnomalyExperiment

s = AnomalyExperiment()

s.setup(
    data,
    categorical_features=categorical,
    numeric_features=continuous + discrete,
    session_id=123,
)

,Description,Value
0,Session id,123
1,Original data shape,"(5000, 37)"
2,Transformed data shape,"(5000, 186)"
3,Numeric features,28
4,Categorical features,9
5,Preprocess,True
6,Imputation type,simple
7,Numeric imputation,mean
8,Categorical imputation,mode
9,Maximum one-hot encoding,-1


# Create Model

In [9]:
s.models()

,Name,Reference
ID,,
abod,Angle-base Outlier Detection,pyod.models.abod.ABOD
cluster,Clustering-Based Local Outlier,pycaret.internal.patches.pyod.CBLOFForceToDouble
cof,Connectivity-Based Local Outlier,pyod.models.cof.COF
iforest,Isolation Forest,pyod.models.iforest.IForest
histogram,Histogram-based Outlier Detection,pyod.models.hbos.HBOS
knn,K-Nearest Neighbors Detector,pyod.models.knn.KNN
lof,Local Outlier Factor,pyod.models.lof.LOF
svm,One-class SVM detector,pyod.models.ocsvm.OCSVM
pca,Principal Component Analysis,pyod.models.pca.PCA


In [9]:
iforest = s.create_model("iforest")
print(iforest)

IForest(behaviour='new', bootstrap=False, contamination=0.05,
    max_features=1.0, max_samples='auto', n_estimators=100, n_jobs=-1,
    random_state=123, verbose=0)


In [10]:
type(iforest)

pyod.models.iforest.IForest

# Analyze Model

In [7]:
s.plot_model(iforest, plot="tsne")

In [8]:
s.plot_model(iforest, plot="umap")

# Assign Model

In [9]:
result = s.assign_model(iforest)
result.head()

,DataUsed_UL,DataUsed_DL,Speed_UL,Speed_DL,LegacyDevicesExist,StationaryVsMobile,WifiScores_top3Devices,PhyRate,SignalStrength,NumClientsConnected,...,RebootPerformedViaCustomer,StationName,NumNeighborsOnSameChannel,NumOverallNeighbors,NumNeighborsOnAdjChannel,NumIPChanges,ExtendedBackhaulRange,EthernetAndNot1Gig,Anomaly,Anomaly_Score
0,224.117294,244.346191,24.886200,170.372864,Yes,Stationary,19.428799,407.362000,-61.579952,7,...,No,2,3,7,0,1,Yes,Yes,0,-0.017529
1,121.534836,190.142715,32.904835,151.496643,Yes,Stationary,22.576670,505.233368,-68.115532,8,...,No,3,6,10,3,0,Yes,No,0,-0.034165
2,106.712860,303.252167,10.293199,59.872658,No,Mobile,22.830999,598.159607,-55.188274,10,...,No,9,6,14,3,3,Yes,No,1,0.010886
3,187.828690,342.273560,69.956963,119.270683,No,Stationary,23.863197,538.285400,-68.701393,4,...,No,3,9,12,5,2,No,Yes,0,-0.019495
4,229.867249,326.287476,27.209818,51.917137,Yes,Stationary,26.248302,688.846924,-56.527905,5,...,Yes,5,10,13,3,1,No,No,0,-0.014380


# Predictions

In [10]:
preds = s.predict_model(iforest, data = data)
preds.head()

,DataUsed_UL,DataUsed_DL,Speed_UL,Speed_DL,LegacyDevicesExist,StationaryVsMobile,WifiScores_top3Devices,PhyRate,SignalStrength,NumClientsConnected,...,StationName_6.0,StationName_1.0,NumNeighborsOnSameChannel,NumOverallNeighbors,NumNeighborsOnAdjChannel,NumIPChanges,ExtendedBackhaulRange,EthernetAndNot1Gig,Anomaly,Anomaly_Score
0,224.117296,244.346184,24.886201,170.372869,1.0,1.0,19.428799,407.362001,-61.579952,7.0,...,0.0,0.0,3.0,7.0,0.0,1.0,1.0,1.0,0,-0.017529
1,121.534834,190.142714,32.904833,151.496648,1.0,1.0,22.576670,505.233358,-68.115528,8.0,...,0.0,0.0,6.0,10.0,3.0,0.0,1.0,0.0,0,-0.034165
2,106.712863,303.252176,10.293199,59.872658,0.0,0.0,22.830999,598.159580,-55.188275,10.0,...,0.0,0.0,6.0,14.0,3.0,3.0,1.0,0.0,1,0.010886
3,187.828692,342.273566,69.956962,119.270685,0.0,1.0,23.863197,538.285427,-68.701393,4.0,...,0.0,0.0,9.0,12.0,5.0,2.0,0.0,1.0,0,-0.019495
4,229.867242,326.287468,27.209818,51.917137,1.0,1.0,26.248302,688.846932,-56.527905,5.0,...,0.0,0.0,10.0,13.0,3.0,1.0,0.0,0.0,0,-0.014380


# Save the model

In [11]:
s.save_model(iforest, "iforest_pipeline")

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['DataUsed_UL', 'DataUsed_DL',
                                              'Speed_UL', 'Speed_DL', 'PhyRate',
                                              'SignalStrength',
                                              'WifiAssociations',
                                              'SON_SteerSuccessStatus',
                                              'KeyProcessCrashes', 'Latency',
                                              'HomeWifiScore',
                                              'PercentageOfDataOn5G',
                                              'ClientBandwidthUsed',
                                              'AirtimeCongestion',
                                              'ActivePhyRate',
                                              'WifiSco...
                  TransformerWrapper(include=['ChannelOfOperation',
                          

# Next Steps
--drafting